# 迷路探索アルゴリズム

## 迷路探索
実コストcost:1ステップ毎に＋1、縦横に移動可能    
推定コスト：マンハッタン距離＝|x1-x2|+|y1-y2|ゴールまでの距離  
スコアscore: score=実コスト+推定コスト

###アルゴリズム


1.   青の壁がある場合には、そちらに移動はできない。元にも戻らない
2.   毎回、左下右上と移動できるかチェック
3.   移動できる場合に、スコアを計算し、最も小さいスコアを選択する
4.   複数の候補があった場合には、競合解消として、左下右上の順とする。  
また同じ方向で複数の候補がある場合には、待ち行列を採用する。



*  既知経路の学習
> ゴールに追いつくことができた場合に、
その経路のコストを毎回 -0.5分減算するようにしてみよう。
これは、うまく行ったケース（今回は経路）を学習して次に活かそうという発想である。数パターンについて、値を付加して平均する



### 実行コード

In [1]:
import numpy as np
from IPython.display import clear_output
import time

class MazeSearch():
  def __init__(self, locate, goal, view_map):
    self.goal = goal
    self.initial_map = view_map
    self.initial_locate = locate

    x, y = locate
    self.view_map = np.array(self.initial_map)
    self.locate = tuple(self.initial_locate)
    self.queue = np.empty((0, 4))
    self.route = [(x, y, 0, 0)]
    self.costmap = np.zeros(self.view_map.shape)

    # B評価：探索方向の優先順位（左→下→右→上）
    self.dx = [-1, 0, 1, 0]
    self.dy = [0, 1, 0, -1]

  # A評価：ヒューリスティック関数（マンハッタン距離）
  def heuristic(self, x, y, cost):
    Ex, Ey = self.goal
    #これまでのコストとマンハッタン距離の和(推定コスト)を返す
    return abs(x - Ex) + abs(y - Ey) + cost

  def search (self, x, y, cost):
    x, y = int(x), int(y)
    Ex, Ey = self.goal
    cost += 1

    # ゴール判定
    if (x, y)==self.goal:
      return

    # 迷路更新/表示
    #スタート地点とすでに通ったことのある場所以外の時に、１(通った印)をつける
    if (x, y)==self.locate or self.view_map[y][x] != 1:
      self.view_map[y][x] = 1
      self.view()


    n = []
    #複数のリスト（やタプルなど）を“まとめて”ループ処理
    for dx, dy in zip(self.dx, self.dy):
      #先ほど定めた優先順位を実装
      nx, ny = x + dx, y + dy
      if self.view_map[ny][nx] == 0:
          score = self.heuristic(nx, ny, cost) + self.costmap[ny][nx]
          n.append([score, nx, ny, cost])

    if len(n) > 0:
      self.queue = np.concatenate([self.queue, n], 0)

    if len(self.queue) == 0:
        return

    q_list = self.queue.tolist()
    q_list.sort(key=lambda val: val[0])

    next_step = q_list.pop(0)

    if len(q_list) == 0:
      self.queue = np.empty((0, 4))
    else:
      self.queue = np.array(q_list)

    score, next_x, next_y, next_cost = next_step

    self.route.append((int(next_x), int(next_y), int(next_cost), score))
    self.search(next_x, next_y, next_cost)



  def run(self, maxlength, learn_rate=0.5, update_costmap=True):
    costmap = np.array(self.costmap)
    for i in range(maxlength):
      # 迷路再設定
      print('\nゴール位置の指定(Ex, Ey)')
      Ex = int(input("Ex="))
      Ey = int(input("Ey="))
      self.goal = Ex, Ey
      self.queue = np.empty((0, 4))
      self.view_map = np.array(self.initial_map)
      self.locate = tuple(self.initial_locate)

      x, y = self.locate
      self.route = [(x, y, 0, 0)]

      self.search(x, y, 0)
      print("ゴール\n")
      goal_route = self.get_goal_route()
      print('ゴールまでの最短経路(x, y, 実コスト, スコア)')
      #ゴールまでの通った最短経路のマスのcostを下げる→通りやすいと判断させる
      for route in goal_route:
        print(route)
        x, y, _, _ = route
        costmap[y][x] -= learn_rate

    if update_costmap:
      costmap = costmap / maxlength
      self.costmap = costmap.tolist()
      print('\n学習後CostMap')
      print(costmap)


  def view(self, on_stop=True):
    clear_output()
    for y, row in enumerate(self.view_map):
      for x, item in enumerate(row):
        if (x, y)==self.goal:
          print("G ", end="")
        elif item == 0:
          print("  ", end="")
        elif item == 1:
          print("o ", end="");
        elif item == 2:
          print("[]", end="")
      print(" ")
    print(" ")
    if on_stop:
      time.sleep(0.5)


  def get_goal_route(self):
      goal_route = []
      for i in reversed(self.route):
        if len(goal_route) > 0:
          if abs(goal_route[0][2] - i[2]) == 1 and abs(goal_route[0][0]-i[0])+abs(goal_route[0][1]-i[1]) == 1:
              goal_route.insert(0, i)
        else:
          goal_route.append(i)
      return goal_route

In [3]:
locate = (2, 2)
goal = (4, 4)
view_map = [
 [2, 2, 2, 2, 2, 2, 2],
 [2, 0, 0, 0, 0, 0, 2],
 [2, 0, 0, 2, 2, 2, 2],
 [2, 0, 2, 2, 0, 0, 2],
 [2, 0, 2, 0, 0, 0, 2],
 [2, 0, 0, 0, 0, 0, 2],
 [2, 2, 2, 2, 2, 2, 2]
]

maze_search = MazeSearch(locate, goal, view_map)

# 実行(学習あり)
print('------学習-------')
maze_search.run(maxlength=3, learn_rate=0.5, update_costmap=True)
# 実行(学習なし)
print('------推論-------')
maze_search.run(maxlength=1, update_costmap=False)

[][][][][][][] 
[]  o o     [] 
[]o o [][][][] 
[]o [][]    [] 
[]o []      [] 
[]o o o o G [] 
[][][][][][][] 
 
ゴール

ゴールまでの最短経路(x, y, 実コスト, スコア)
(2, 2, 0, 0)
(1, 2, 1, 7.666666666666667)
(1, 3, 2, 7.666666666666667)
(1, 4, 3, 7.666666666666667)
(1, 5, 4, 7.666666666666667)
(2, 5, 5, 7.666666666666667)
(3, 5, 6, 7.666666666666667)
(4, 5, 7, 8.0)
(5, 5, 8, 8.0)


学習において、スコアが増えない瞬間

→学習済みの経路を通っているということ

costMap→今まででゴールにたどり着けた「いい感じの道」

0：未学習

負：良い道（通るとスコアが下がる）通った回数が多いマスほど何度も減算されるためより小さくなる

正：悪い道（今回は使っていない）


実コストは +1 ずつ増えている
一方でスコア(推定コスト)はあまり増えていない
→ 評価値が相殺されている

これは A評価（学習あり）でしか起きない現象である。
↓
B評価の方向の優先順位のみの場合は学習ではなく単に距離が同じだった場合の競合解消として行うだけ

① スコアの推移

B：単調増加

A：学習により抑制・逆転が起きる

② 再実行時の安定性

B：毎回同じ探索

A：良い経路が優先される

③ 数値の性質

B：整数中心

A：小数を含む（学習の影響）

→成功経路の cost が減少する際平均化されるため、小数値が生じる